## 1. Imports

In [1]:
import networkx as nx
import pandas as pd
import numpy as np

from IPython.display import display

## 2. Directed graph

In [2]:
def load_directed_graph(edges_path, labels_path=None):
    """
    Loads Cora as a directed and unweighted citation graph.

    IMPORTANT:
    The Cora edge file is interpreted as:
        cited_paper, citing_paper, weight

    Therefore, each row is converted to:
        citing_paper -> cited_paper

    Exact duplicate directed edges are removed.
    Reciprocal citations are kept as two distinct directed edges.
    Self-loops are removed.
    """

    raw_edges_df = pd.read_csv(
        edges_path,
        sep=",",
        header=None,
        names=["cited", "citing", "weight"]
    )

    raw_edges_df["cited"] = raw_edges_df["cited"].astype(int)
    raw_edges_df["citing"] = raw_edges_df["citing"].astype(int)

    original_rows = len(raw_edges_df)

    # Remove self-loops.
    raw_edges_df = raw_edges_df[
        raw_edges_df["cited"] != raw_edges_df["citing"]
    ].copy()

    removed_self_loops = original_rows - len(raw_edges_df)

    # Remove only exact duplicate citation records.
    # Reciprocal relationships remain distinct.
    before_duplicates = len(raw_edges_df)

    raw_edges_df = (
        raw_edges_df
        .drop_duplicates(subset=["cited", "citing"])
        .reset_index(drop=True)
    )

    removed_duplicates = before_duplicates - len(raw_edges_df)

    # Keep the graph unweighted.
    if not np.allclose(raw_edges_df["weight"].astype(float), 1.0):
        print(
            "Warning: weights different from 1 were found. "
            "They will not be used."
        )

    # Standardize the cleaned edge list to the actual graph direction:
    # source = citing paper
    # target = cited paper
    directed_edges_df = pd.DataFrame({
        "source": raw_edges_df["citing"].astype(int),
        "target": raw_edges_df["cited"].astype(int),
        "weight": 1
    })

    directed_edges_df = (
        directed_edges_df
        .sort_values(["source", "target"])
        .reset_index(drop=True)
    )

    G = nx.DiGraph()

    # Add labelled nodes first so node ordering is deterministic.
    if labels_path is not None:
        labels_df = pd.read_csv(
            labels_path,
            sep=",",
            header=None,
            names=["node", "label"]
        )

        labels_df["node"] = labels_df["node"].astype(int)
        labels_df["label"] = labels_df["label"].astype(int)

        G.add_nodes_from(
            sorted(labels_df["node"].unique().tolist())
        )

    # Correct citation direction: citing -> cited.
    G.add_edges_from(
        directed_edges_df[["source", "target"]]
        .itertuples(index=False, name=None)
    )

    print("=== Directed graph loading ===")
    print("Raw Cora order: cited, citing, weight")
    print("Graph direction used: citing -> cited")
    print(f"Rows in original edge file: {original_rows}")
    print(f"Self-loops removed: {removed_self_loops}")
    print(f"Exact directed duplicates removed: {removed_duplicates}")
    print(f"Directed edges retained: {G.number_of_edges()}")

    return G, directed_edges_df


## 3. Classes

In [3]:
def load_labels_as_circles(labels_path):
    labels_df = pd.read_csv(
        labels_path,
        sep=",",
        header=None,
        names=["node", "label"]
    )

    labels_df["node"] = labels_df["node"].astype(int)
    labels_df["label"] = labels_df["label"].astype(int)

    label_circles = {}

    for label, group in labels_df.groupby("label", sort=True):
        label_circles[f"class_{label}"] = sorted(
            group["node"].tolist()
        )

    return label_circles, labels_df

## 4. Functions

In [4]:
def summarize_circles(G, circles_dict):
    node_ids = sorted(G.nodes())
    node_idx = {
        node: i for i, node in enumerate(node_ids)
    }

    circle_names = list(circles_dict.keys())

    membership_array = np.zeros(
        (len(node_ids), len(circle_names)),
        dtype=int
    )

    for j, circle_name in enumerate(circle_names):
        for node in circles_dict[circle_name]:
            node = int(node)

            if node in node_idx:
                membership_array[node_idx[node], j] = 1

    num_memberships = membership_array.sum(axis=1)

    print("=== Circle/Class Summary ===")
    print(f"Total classes: {len(circles_dict)}")
    print(
        "Nodes in at least one class:",
        int((num_memberships > 0).sum())
    )
    print(
        "Nodes in multiple classes:",
        int((num_memberships > 1).sum())
    )
    print(
        "Nodes not in any class:",
        int((num_memberships == 0).sum())
    )

    return membership_array


def summarize_label_groups(G, circles_dict):
    total_nodes = G.number_of_nodes()

    summary_df = pd.DataFrame.from_dict(
        {
            class_name: len(members)
            for class_name, members in circles_dict.items()
        },
        orient="index",
        columns=["num_nodes"]
    )

    summary_df["percent_nodes"] = (
        summary_df["num_nodes"] / total_nodes * 100
    ).round(2)

    summary_df = summary_df.sort_values(
        "num_nodes",
        ascending=False
    )

    print("=== Label/Class Distribution ===")
    display(summary_df)

    return summary_df

## 5. Membership matrix

In [5]:
def compute_membership_matrix(G, circles_dict):
    node_ids = sorted(G.nodes())
    circle_names = list(circles_dict.keys())

    membership_matrix = pd.DataFrame(
        0,
        index=node_ids,
        columns=circle_names,
        dtype=int
    )

    for circle_name, members in circles_dict.items():
        valid_members = [
            int(node)
            for node in members
            if int(node) in membership_matrix.index
        ]

        membership_matrix.loc[
            valid_members,
            circle_name
        ] = 1

    membership_matrix.index.name = "node"

    return membership_matrix

## 6. ARD 

In [6]:
def compute_outgoing_ard_matrix(G, circles_dict):
    """
    Computes outgoing ARD.

    ard_matrix.loc[i, class_k] is the number of outgoing edges
    i -> j for which node j belongs to class_k.
    """

    node_ids = sorted(G.nodes())
    circle_names = list(circles_dict.keys())

    ard_matrix = pd.DataFrame(
        0,
        index=node_ids,
        columns=circle_names,
        dtype=int
    )

    circle_sets = {
        circle_name: set(map(int, members))
        for circle_name, members in circles_dict.items()
    }

    for node in node_ids:
        # Successors are the nodes cited by 'node'.
        outgoing_neighbors = set(G.successors(node))

        for circle_name in circle_names:
            ard_matrix.at[node, circle_name] = len(
                outgoing_neighbors.intersection(
                    circle_sets[circle_name]
                )
            )

    ard_matrix.index.name = "node"

    return ard_matrix

In [7]:
def validate_outgoing_ard(
    G,
    ard_matrix,
    membership_matrix
):
    """
    If classes are mutually exclusive and cover every node,
    each ARD row sum must equal the node out-degree.
    """

    out_degrees = pd.Series(
        dict(G.out_degree()),
        name="out_degree",
        dtype=int
    ).reindex(ard_matrix.index, fill_value=0)

    ard_sums = ard_matrix.sum(axis=1).astype(int)
    ard_sums.name = "ard_row_sum"

    membership_counts = membership_matrix.sum(axis=1)

    all_nodes_have_one_class = bool(
        (membership_counts == 1).all()
    )

    comparison = pd.concat(
        [out_degrees, ard_sums],
        axis=1
    )

    comparison["difference"] = (
        comparison["ard_row_sum"]
        - comparison["out_degree"]
    )

    matches = bool(
        (comparison["difference"] == 0).all()
    )

    print("=== Outgoing ARD validation ===")
    print(
        "Every node belongs to exactly one class:",
        all_nodes_have_one_class
    )
    print(
        "ARD row sums equal out-degrees:",
        matches
    )
    print(
        "Number of mismatching nodes:",
        int((comparison["difference"] != 0).sum())
    )

    if not matches:
        print("\nFirst mismatches:")
        display(
            comparison[
                comparison["difference"] != 0
            ].head(10)
        )

    return matches, comparison

## 7. Compute `total_prop`

In [8]:
def compute_total_prop_outgoing(
    G,
    circles_dict,
    membership_matrix
):
    """
    Computes the total proportion of outgoing ties represented
    by the ARD groups.

    For each directed citation i -> j, only the membership of
    the cited/target node j is counted.
    """

    total_directed_edges = G.number_of_edges()
    circle_names = list(circles_dict.keys())

    target_ties = pd.Series(
        0,
        index=circle_names,
        dtype=int
    )

    for _, target in G.edges():
        target_membership = membership_matrix.loc[target]

        target_ties = target_ties.add(
            target_membership,
            fill_value=0
        )

    target_ties = target_ties.astype(int)

    if total_directed_edges == 0:
        total_prop = 0.0
    else:
        total_prop = (
            target_ties.sum()
            / total_directed_edges
        )

    # Explicitly name the column to avoid KeyError.
    target_ties_df = target_ties.to_frame(
        name="target_ties"
    )

    if total_directed_edges > 0:
        target_ties_df["proportion_of_directed_edges"] = (
            target_ties_df["target_ties"]
            / total_directed_edges
        )
    else:
        target_ties_df["proportion_of_directed_edges"] = 0.0

    print("=== Outgoing total.prop ===")
    print(
        f"Directed edges: {total_directed_edges}"
    )
    print(
        f"Sum of target-group counts: {target_ties.sum()}"
    )
    print(f"Total prop: {total_prop}")

    if np.isclose(total_prop, 1.0):
        print(
            "Check passed: each directed edge is represented "
            "once by the target classes."
        )
    else:
        print(
            "Warning: total_prop is not 1. "
            "Check missing or overlapping classes."
        )

    return total_prop, target_ties_df


## 8. Data

In [9]:
edges_path = "cora.edges"
labels_path = "cora.node_labels"

G, directed_edges_df = load_directed_graph(
    edges_path,
    labels_path
)

label_circles, labels_df = load_labels_as_circles(
    labels_path
)

print("\n=== Directed Graph Info ===")
print("Citation orientation: citing -> cited")
print(f"Directed graph: {G.is_directed()}")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Directed edges: {G.number_of_edges()}")
print(f"Self-loops: {nx.number_of_selfloops(G)}")

weak_components = list(
    nx.weakly_connected_components(G)
)

strong_components = list(
    nx.strongly_connected_components(G)
)

print(
    "Weakly connected components:",
    len(weak_components)
)
print(
    "Largest weakly connected component:",
    len(max(weak_components, key=len))
)
print(
    "Strongly connected components:",
    len(strong_components)
)
print(
    "Largest strongly connected component:",
    len(max(strong_components, key=len))
)


=== Directed graph loading ===
Raw Cora order: cited, citing, weight
Graph direction used: citing -> cited
Rows in original edge file: 5429
Self-loops removed: 0
Exact directed duplicates removed: 0
Directed edges retained: 5429

=== Directed Graph Info ===
Citation orientation: citing -> cited
Directed graph: True
Nodes: 2708
Directed edges: 5429
Self-loops: 0
Weakly connected components: 78
Largest weakly connected component: 2485
Strongly connected components: 2526
Largest strongly connected component: 13


## 9. Metrics

In [10]:
def compute_directed_graph_metrics(G):
    """
    Computes structural properties of a directed graph.

    Directed path metrics are calculated on the largest strongly
    connected component.

    For comparison with the previous undirected analysis, path
    metrics are also calculated on the largest weakly connected
    component after temporarily ignoring edge direction.
    """

    if G.number_of_nodes() == 0:
        raise ValueError("The graph has no nodes.")

    node_ids = sorted(G.nodes())

    in_degree_values = np.array(
        [G.in_degree(node) for node in node_ids]
    )

    out_degree_values = np.array(
        [G.out_degree(node) for node in node_ids]
    )

    total_degree_values = (
        in_degree_values + out_degree_values
    )

    weak_components = list(
        nx.weakly_connected_components(G)
    )

    strong_components = list(
        nx.strongly_connected_components(G)
    )

    largest_weak_nodes = max(
        weak_components,
        key=len
    )

    largest_strong_nodes = max(
        strong_components,
        key=len
    )

    G_largest_weak = G.subgraph(
        largest_weak_nodes
    ).copy()

    G_largest_strong = G.subgraph(
        largest_strong_nodes
    ).copy()

    # Used only for metrics comparable with the old undirected version.
    G_largest_weak_undirected = (
        G_largest_weak.to_undirected()
    )

    if G_largest_strong.number_of_nodes() == 1:
        directed_diameter = 0
        directed_average_path = 0.0
    else:
        directed_diameter = nx.diameter(
            G_largest_strong
        )

        directed_average_path = (
            nx.average_shortest_path_length(
                G_largest_strong
            )
        )

    if G_largest_weak_undirected.number_of_nodes() == 1:
        weak_undirected_diameter = 0
        weak_undirected_average_path = 0.0
    else:
        weak_undirected_diameter = nx.diameter(
            G_largest_weak_undirected
        )

        weak_undirected_average_path = (
            nx.average_shortest_path_length(
                G_largest_weak_undirected
            )
        )

    reciprocal_edge_pairs = (
        sum(
            1
            for source, target in G.edges()
            if G.has_edge(target, source)
        )
        // 2
    )

    reciprocity = nx.reciprocity(G)

    if reciprocity is None:
        reciprocity = np.nan

    try:
        out_in_assortativity = (
            nx.degree_assortativity_coefficient(
                G,
                x="out",
                y="in"
            )
        )
    except Exception:
        out_in_assortativity = np.nan

    metrics = {
        "Number of nodes": G.number_of_nodes(),
        "Number of directed edges": G.number_of_edges(),
        "Average in-degree": in_degree_values.mean(),
        "Average out-degree": out_degree_values.mean(),
        "Maximum in-degree": in_degree_values.max(),
        "Maximum out-degree": out_degree_values.max(),
        "Nodes with zero in-degree": int(
            (in_degree_values == 0).sum()
        ),
        "Nodes with zero out-degree": int(
            (out_degree_values == 0).sum()
        ),
        "Number of isolated nodes": int(
            (total_degree_values == 0).sum()
        ),
        "Number of weakly connected components": len(
            weak_components
        ),
        "Largest weakly connected component": (
            G_largest_weak.number_of_nodes()
        ),
        "Number of strongly connected components": len(
            strong_components
        ),
        "Largest strongly connected component": (
            G_largest_strong.number_of_nodes()
        ),
        "Directed graph density": nx.density(G),
        "Reciprocal edge pairs": reciprocal_edge_pairs,
        "Reciprocity": reciprocity,
        "Average directed clustering coefficient": (
            nx.average_clustering(G)
        ),
        "Out-in degree assortativity": (
            out_in_assortativity
        ),
        "Diameter of largest strong component": (
            directed_diameter
        ),
        "Average directed path length in largest strong component": (
            directed_average_path
        ),
        "Diameter of largest weak component (direction ignored)": (
            weak_undirected_diameter
        ),
        "Average path length in largest weak component (direction ignored)": (
            weak_undirected_average_path
        ),
        "Weakly connected graph": nx.is_weakly_connected(G),
        "Strongly connected graph": nx.is_strongly_connected(G)
    }

    metrics_df = pd.DataFrame(
        {
            "Metric": list(metrics.keys()),
            "Value": list(metrics.values())
        }
    )

    metrics_df["Value"] = metrics_df["Value"].apply(
        lambda value: round(float(value), 4)
        if isinstance(value, (float, np.floating))
        and not np.isnan(value)
        else value
    )

    return metrics_df


cora_directed_graph_metrics = (
    compute_directed_graph_metrics(G)
)

display(cora_directed_graph_metrics)

,Metric,Value
0,Number of nodes,2708
1,Number of directed edges,5429
2,Average in-degree,2.0048
3,Average out-degree,2.0048
4,Maximum in-degree,166
5,Maximum out-degree,5
6,Nodes with zero in-degree,1143
7,Nodes with zero out-degree,486
8,Number of isolated nodes,0
9,Number of weakly connected components,78


## 10. Summary of classes

In [11]:
membership_array_original = summarize_circles(
    G,
    label_circles
)

circle_summary_df = summarize_label_groups(
    G,
    label_circles
)

=== Circle/Class Summary ===
Total classes: 7
Nodes in at least one class: 2708
Nodes in multiple classes: 0
Nodes not in any class: 0
=== Label/Class Distribution ===


,num_nodes,percent_nodes
class_3,818,30.21
class_4,426,15.73
class_2,418,15.44
class_7,351,12.96
class_1,298,11.00
class_5,217,8.01
class_6,180,6.65


## 11. Construction of the ARD

In [12]:
membership_matrix = compute_membership_matrix(
    G,
    label_circles
)

ard_matrix_outgoing = compute_outgoing_ard_matrix(
    G,
    label_circles
)

print("=== Outgoing ARD Matrix Info ===")
print(
    f"ARD dimensions: {ard_matrix_outgoing.shape}"
)
print(
    f"Number of classes used: {len(label_circles)}"
)

ard_is_valid, ard_validation_df = (
    validate_outgoing_ard(
        G,
        ard_matrix_outgoing,
        membership_matrix
    )
)

display(ard_matrix_outgoing.head())

=== Outgoing ARD Matrix Info ===
ARD dimensions: (2708, 7)
Number of classes used: 7
=== Outgoing ARD validation ===
Every node belongs to exactly one class: True
ARD row sums equal out-degrees: True
Number of mismatching nodes: 0


,class_1,class_2,class_3,class_4,class_5,class_6,class_7
node,,,,,,,
1,0,0,2,0,0,0,0
2,0,0,0,0,0,1,0
3,0,0,0,0,4,0,0
4,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0


In [13]:
total_prop_outgoing, target_ties_df = (
    compute_total_prop_outgoing(
        G,
        label_circles,
        membership_matrix
    )
)

display(target_ties_df)

=== Outgoing total.prop ===
Directed edges: 5429
Sum of target-group counts: 5429
Total prop: 1.0
Check passed: each directed edge is represented once by the target classes.


,target_ties,proportion_of_directed_edges
class_1,521,0.095966
class_2,935,0.172223
class_3,1463,0.269479
class_4,840,0.154725
class_5,535,0.098545
class_6,313,0.057653
class_7,822,0.151409


In [14]:
assert G.is_directed(), (
    "The graph should be directed."
)

assert nx.number_of_selfloops(G) == 0, (
    "Self-loops were found."
)

assert ard_is_valid, (
    "The outgoing ARD row sums do not match "
    "the out-degrees."
)

assert np.isclose(total_prop_outgoing, 1.0), (
    "total_prop should be 1 when classes are "
    "mutually exclusive and cover all target nodes."
)

# Since the seven Cora classes are exclusive and exhaustive,
# every outgoing citation must be counted exactly once in the ARD.
assert int(ard_matrix_outgoing.to_numpy().sum()) == G.number_of_edges(), (
    "The total number of ARD counts must equal the number "
    "of directed citation edges."
)

print("All final checks passed.")
print("Citation direction confirmed: citing -> cited")
print(
    "Each ARD row represents the classes of papers cited "
    "by that node."
)


All final checks passed.
Citation direction confirmed: citing -> cited
Each ARD row represents the classes of papers cited by that node.


## 12. Save results

In [15]:
# Save the outgoing ARD.
ard_matrix_outgoing.to_csv(
    "ard_matrix_cora_directed_out.csv"
)

# Save node memberships and class information.
membership_matrix.to_csv(
    "membership_matrix_cora_directed.csv"
)

circle_summary_df.to_csv(
    "circle_summary_cora_directed.csv"
)

labels_df.to_csv(
    "labels_cora_directed.csv",
    index=False
)

# Save directed graph metrics.
cora_directed_graph_metrics.to_csv(
    "cora_directed_network_metrics.csv",
    index=False
)

# Save target-class tie counts.
target_ties_df.to_csv(
    "cora_directed_target_ties.csv"
)

# Save the cleaned citation edge list in the actual graph direction:
# source = citing paper, target = cited paper.
directed_edges_to_save = directed_edges_df[
    ["source", "target", "weight"]
].copy()

directed_edges_to_save.to_csv(
    "cora_directed_edges.csv",
    index=False,
    header=False
)

# Save total.prop.
with open(
    "total_prop_cora_directed_out.txt",
    "w"
) as file:
    file.write(
        f"Total Proportion: {total_prop_outgoing}\n"
    )

print("Files saved successfully:")
print("- ard_matrix_cora_directed_out.csv")
print("- membership_matrix_cora_directed.csv")
print("- circle_summary_cora_directed.csv")
print("- labels_cora_directed.csv")
print("- cora_directed_network_metrics.csv")
print("- cora_directed_target_ties.csv")
print("- cora_directed_edges.csv")
print("- total_prop_cora_directed_out.txt")


Files saved successfully:
- ard_matrix_cora_directed_out.csv
- membership_matrix_cora_directed.csv
- circle_summary_cora_directed.csv
- labels_cora_directed.csv
- cora_directed_network_metrics.csv
- cora_directed_target_ties.csv
- cora_directed_edges.csv
- total_prop_cora_directed_out.txt
